# Fraud Mitigation Agent · 01 Prompt Agent

Primer paso: responder con un proveedor local determinístico. El LLM no decide el riesgo.


### Recordemos la infraestructura del notebook 00

Base de datos en memoria, configuración y datos sintéticos, en una sola celda para que este notebook corra solo.

In [ ]:
import os  # para leer variables de entorno (Colab Secrets ya copiados acá)
import copy  # para copiar documentos sin compartir referencias (ver InMemoryCollection)
import uuid  # para generar ids únicos de documentos insertados
from dataclasses import dataclass, field  # dataclass: clases con campos, sin escribir __init__ a mano
from typing import Any, Optional  # Optional[str] = "puede ser str o None"


@dataclass(frozen=True)  # frozen=True: una vez creado, Settings no se puede modificar
class Settings:
    """Configuración central del workshop, cargada desde variables de entorno o Colab Secrets."""
    mongodb_uri: Optional[str] = None  # None si no configuraste Atlas: usamos memoria local
    database_name: str = "fraud_mitigation_agent_workshop"  # nombre de la base en Atlas
    source_tag: str = "fraud_mitigation_agent_colab_workshop"  # para poder borrar solo lo nuestro
    embedding_dimensions: int = 8  # tamaño de los vectores que genera deterministic_embedding

    @classmethod
    def from_env(cls, **overrides):
        # Lee cada valor de una variable de entorno, con un default si no existe.
        values = {
            "mongodb_uri": os.getenv("MONGODB_URI"),
            "database_name": os.getenv("FRAUD_MITIGATION_AGENT_DATABASE", "fraud_mitigation_agent_workshop"),
            "source_tag": os.getenv("FRAUD_MITIGATION_AGENT_SOURCE_TAG", "fraud_mitigation_agent_colab_workshop"),
            "embedding_dimensions": int(os.getenv("FRAUD_MITIGATION_AGENT_EMBEDDING_DIMENSIONS", "8")),
        }
        values.update(overrides)  # permite pisar cualquier valor a mano, ej. Settings.from_env(database_name="test")
        return cls(**values)  # construye el Settings con esos valores


# --- Base de datos en memoria: imita la interfaz de pymongo (find_one, find,
# replace_one, delete_many, insert_one) para que el workshop corra sin Atlas. ---

def _matches(doc, query):
    # Recorre cada condición del query y las compara contra el documento.
    for key, expected in query.items():
        actual = doc.get(key)
        if isinstance(expected, dict):
            # Soporta los dos operadores de Mongo que este workshop necesita:
            # $exists (¿tiene o no tiene la clave?) y $in (¿está en esta lista?).
            if "$exists" in expected and (key in doc) != bool(expected["$exists"]):
                return False
            if "$in" in expected and actual not in expected["$in"]:
                return False
        elif actual != expected:
            # Caso simple: {"campo": valor} exige igualdad exacta.
            return False
    return True  # ninguna condición falló: el documento matchea


def _project(row, projection):
    item = copy.deepcopy(row)  # copia defensiva: nunca devolvemos el documento original
    if not projection:
        return item  # sin projection, se devuelve el documento completo
    excludes = [key for key, value in projection.items() if value == 0]  # claves a quitar
    includes = [key for key, value in projection.items() if value == 1]  # claves a conservar
    if includes:
        return {key: item[key] for key in includes if key in item}  # solo las incluidas
    for key in excludes:
        item.pop(key, None)  # saca las excluidas (típicamente {"_id": 0})
    return item


class _InsertResult:
    def __init__(self, inserted_id):
        self.inserted_id = inserted_id  # imita el objeto que devuelve pymongo.insert_one


class InMemoryCollection:
    def __init__(self):
        self.rows = []  # todos los documentos de esta "colección" viven en esta lista

    def find_one(self, query, projection=None):
        for row in self.rows:
            if _matches(row, query):
                return _project(row, projection)  # devuelve el primero que matchea
        return None  # ninguno matcheó

    def find(self, query=None, projection=None):
        query = query or {}  # sin query, devuelve todos los documentos
        return [_project(row, projection) for row in self.rows if _matches(row, query)]

    def replace_one(self, query, replacement, upsert=False):
        for i, row in enumerate(self.rows):
            if _matches(row, query):
                self.rows[i] = copy.deepcopy(replacement)  # ya existía: lo reemplaza
                return
        if upsert:
            self.rows.append(copy.deepcopy(replacement))  # no existía: lo agrega (upsert)

    def delete_many(self, query):
        self.rows = [row for row in self.rows if not _matches(row, query)]  # se queda con lo que NO matchea

    def insert_one(self, document):
        item = copy.deepcopy(document)  # copia defensiva del documento a insertar
        item.setdefault("_id", uuid.uuid4().hex)  # le pone un _id si no traía uno
        self.rows.append(item)
        return _InsertResult(item["_id"])

    def aggregate(self, pipeline):
        # $vectorSearch no está disponible en memoria local a propósito:
        # en el notebook 06 vamos a manejar esto con un fallback de similitud
        # coseno calculado en Python.
        raise RuntimeError("Atlas aggregation unavailable in local memory mode")


class InMemoryDB:
    """Imita `client[database_name]` / `db.coleccion` de pymongo, creando
    colecciones sobre la marcha la primera vez que se acceden."""

    def __init__(self):
        self._collections = {}  # diccionario nombre -> InMemoryCollection

    def __getitem__(self, name):
        return self.__getattr__(name)  # db["transactions"] hace lo mismo que db.transactions

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)  # evita interceptar atributos internos como _collections
        self._collections.setdefault(name, InMemoryCollection())  # la crea si es la primera vez
        return self._collections[name]


def get_client(uri, timeout_ms=10000):
    """Conecta a MongoDB Atlas real. Solo se usa si defines MONGODB_URI."""
    from pymongo import MongoClient  # import perezoso: solo hace falta si de verdad usás Atlas
    if not uri:
        raise ValueError("MONGODB_URI is required")
    client = MongoClient(uri, serverSelectionTimeoutMS=timeout_ms)
    client.admin.command("ping")  # falla rápido acá si la conexión no funciona
    return client


def get_database(client, database_name):
    return client[database_name]  # selecciona (o crea) la base dentro del cluster


import hashlib  # para generar un hash reproducible de cada palabra
import re  # para separar el texto en palabras (tokens)
import numpy as np  # para operar con vectores (sumas, norma)


def _token_value(token, dimensions):
    digest = hashlib.sha256(token.encode("utf-8")).digest()  # 32 bytes, siempre iguales para el mismo token
    values = np.frombuffer(digest, dtype=np.uint8)[:dimensions].astype(float)  # toma los primeros `dimensions` bytes
    return (values / 127.5) - 1.0  # reescala de [0, 255] a, aproximadamente, [-1, 1]


def deterministic_embedding(text, dimensions=8):
    """Embedding determinístico para el workshop (hashing, sin modelo ni red).
    Lo usamos desde ya para poder sembrar los datos de ejemplo; en el
    notebook 06 vamos a entender cómo funciona y a construir búsqueda por
    similitud vectorial sobre él."""
    tokens = re.findall(r"[a-zA-Z0-9_áéíóúñ-]+", (text or "").lower())  # separa el texto en palabras, en minúscula
    if not tokens:
        return [0.0] * dimensions  # texto vacío -> vector de ceros
    vector = np.zeros(dimensions, dtype=float)  # arranca en cero
    for token in tokens:
        vector += _token_value(token, dimensions)  # suma el vector de cada palabra
    norm = np.linalg.norm(vector)  # longitud del vector resultante
    if norm == 0:
        return [0.0] * dimensions  # evita dividir por cero
    return (vector / norm).round(6).tolist()  # normaliza (longitud 1) y lo convierte a lista de Python


def _pattern(tx_id, fraud_type, text):
    return {
        "tx_id": tx_id,
        "fraud_confirmed": True,
        "fraud_type": fraud_type,
        "fraud_signature_text": text,
        "embedding": deterministic_embedding(text),
        "source_tag": "fraud_mitigation_agent_synthetic",
    }


def demo_documents():
    # Cuatro "firmas" de fraude conocidas: la colección fraud_patterns que la
    # búsqueda por similitud vectorial va a comparar contra cada transacción nueva.
    patterns = [
        _pattern("pattern-001", "account_takeover", "new device new ip impossible travel odd hour credential reset high amount"),
        _pattern("pattern-002", "card_testing", "many small attempts new ip repeated velocity web checkout"),
        _pattern("pattern-003", "synthetic_identity", "new customer device mismatch unusual geo high amount mobile"),
        _pattern("pattern-004", "money_mule", "rapid transfer beneficiary new device distant geo unusual hour"),
    ]
    # Tres arquetipos de cliente, cada uno emparejado con una transacción que
    # se espera que caiga en una banda de decisión distinta (APPROVE/STEP-UP/DENY).
    customers = [
        {
            "customer_id": "customer-normal",
            "usual_ips": ["198.51.100.10"],
            "usual_devices": ["device-normal-001"],
            "usual_countries": ["MX"],
            "avg_amount": 1200,
            "p95_amount": 4500,
            "transactions_24h": 3,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "customer_id": "customer-stepup",
            "usual_ips": ["198.51.100.20"],
            "usual_devices": ["device-step-001"],
            "usual_countries": ["MX"],
            "avg_amount": 1800,
            "p95_amount": 9000,
            "transactions_24h": 4,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "customer_id": "customer-risky",
            "usual_ips": ["198.51.100.30"],
            "usual_devices": ["device-risky-001"],
            "usual_countries": ["MX"],
            "avg_amount": 2500,
            "p95_amount": 12000,
            "transactions_24h": 2,
            "transactions_10m": 1,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
    ]
    # tx-normal-001 -> se espera APPROVE, tx-stepup-001 -> se espera STEP-UP,
    # tx-risky-001 -> se espera DENY. Los usamos en todos los notebooks.
    transactions = [
        {
            "tx_id": "tx-normal-001", "customer_id": "customer-normal", "amount": 850,
            "currency": "MXN", "timestamp": "2025-01-15T16:20:00Z", "channel": "web",
            "ip": "198.51.100.10", "device_id": "device-normal-001", "geo_km_from_usual": 2,
            "ground_truth_fraud": False, "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "tx_id": "tx-stepup-001", "customer_id": "customer-stepup", "amount": 12000,
            "currency": "MXN", "timestamp": "2025-01-15T22:40:00Z", "channel": "mobile",
            "ip": "198.51.100.20", "device_id": "device-step-001", "geo_km_from_usual": 120,
            "ground_truth_fraud": False, "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "tx_id": "tx-risky-001", "customer_id": "customer-risky", "amount": 5000000,
            "currency": "COP", "timestamp": "2025-01-15T03:00:00Z", "channel": "mobile",
            "ip": "203.0.113.30", "device_id": "device-new-003", "geo_km_from_usual": 9000,
            "ground_truth_fraud": True, "source_tag": "fraud_mitigation_agent_synthetic",
        },
    ]
    rules = {
        "config_id": "risk_rules_config",
        "version": "demo-v1",
        "enabled": True,
        "thresholds": {
            "high_amount": 1000000,
            "amount_multiplier": 5,
            "impossible_travel_km": 500,
            "velocity_10m": 5,
        },
        "weights": {"vector": 0.40, "signals": 0.35, "rules": 0.25},
        "decision_thresholds": {"approve_max": 39, "step_up_max": 69},
        "source_tag": "fraud_mitigation_agent_synthetic",
    }
    return {"patterns": patterns, "customers": customers, "transactions": transactions, "rules": rules}


def seed_demo_data(db, reset=False):
    """Carga los datos sintéticos en la base (Atlas o InMemoryDB). Es idempotente:
    replace_one(upsert=True) evita duplicados si vuelves a correr esta celda."""
    docs = demo_documents()  # arma los 4 patrones, 3 clientes, 3 transacciones y las reglas
    collections = {  # atajos a cada colección de destino
        "patterns": db["fraud_patterns"],
        "customers": db["customer_state"],
        "transactions": db["transactions"],
        "rules": db["risk_rules_config"],
    }
    if reset:
        for collection in collections.values():
            # Borra solo lo que sembramos nosotros (por source_tag), nunca datos ajenos.
            collection.delete_many({"source_tag": {"$in": ["fraud_mitigation_agent_synthetic", "fraud_mitigation_agent_colab_workshop"]}})
    for document in docs["patterns"]:
        collections["patterns"].replace_one({"tx_id": document["tx_id"]}, document, upsert=True)  # inserta o actualiza
    for document in docs["customers"]:
        collections["customers"].replace_one({"customer_id": document["customer_id"]}, document, upsert=True)
    for document in docs["transactions"]:
        document = dict(document)  # copia: no modificamos el diccionario original de demo_documents()
        document["fraud_signature_text"] = (
            "new device new ip impossible travel odd hour high amount"
            if document["ground_truth_fraud"] else  # las transacciones fraudulentas comparten esta descripción...
            "familiar device familiar ip normal amount"  # ...y las normales, esta otra
        )
        document["embedding"] = deterministic_embedding(document["fraud_signature_text"])  # vector para vector search
        collections["transactions"].replace_one({"tx_id": document["tx_id"]}, document, upsert=True)
    collections["rules"].replace_one({"config_id": docs["rules"]["config_id"]}, docs["rules"], upsert=True)
    # Resumen de cuántos documentos se sembraron en cada colección (listas) o 1 (las reglas, un solo documento).
    return {key: len(value) if isinstance(value, list) else 1 for key, value in docs.items()}


# Los Secrets de Colab no se inyectan solos como variables de entorno: hay
# que leerlos explícitamente con userdata.get(...) y copiarlos a os.environ.
# Cada clave se intenta por separado para que una que no exista (userdata.get
# lanza una excepción, no devuelve None) no tumbe la lectura de las demás.
try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    for _secret_name in ("MONGODB_URI", "LLM_API_KEY", "LLM_MODEL", "LLM_BASE_URL"):
        try:
            _secret_value = userdata.get(_secret_name)
        except Exception:
            _secret_value = None
        if _secret_value:
            os.environ[_secret_name] = _secret_value

settings = Settings.from_env()
if settings.mongodb_uri:
    # pymongo no viene preinstalado en este notebook (a diferencia de 00, que
    # sí lo instala siempre): solo lo instalamos aquí, justo a tiempo, si de
    # verdad vas a usar Atlas real. El camino en memoria no lo necesita.
    try:
        import pymongo  # noqa: F401
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo[srv]"], check=True)
    client = get_client(settings.mongodb_uri)
    db = get_database(client, settings.database_name)
else:
    db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


### El agente (etapa 1 de 8): solo responde preguntas

In [ ]:
class MockLLMProvider:
    """Proveedor offline para Colab: determinístico, sin red y sin API key."""

    def complete(self, prompt, system=None):
        # Coincidencia de palabras clave en vez de un modelo real: alcanza
        # para demostrar la mecánica del agente sin necesitar ninguna API key.
        text = (prompt or "").lower()  # normaliza el prompt para comparar en minúscula
        if "fraud mitigation agent" in text or "workshop" in text:
            return "El Fraud Mitigation Agent separa herramientas de contexto, scoring determinístico y decisión auditable."
        if "fraude" in text or "fraud" in text:
            return "El flujo combina reglas, señales de comportamiento y similitud vectorial; el resultado final no depende de una respuesta libre del LLM."
        return "MockLLMProvider: respuesta local reproducible para el workshop."  # respuesta genérica de reserva


class FraudAgent:
    """Etapa 1 de 8: el agente solo sabe responder preguntas con un LLM (mock).
    Todavía no tiene herramientas ni analiza transacciones."""

    def __init__(self, db, provider=None):
        self.db = db  # todavía no se usa en esta etapa, pero ya lo guardamos
        # MockLLMProvider por defecto: cero API keys, cero llamadas de red.
        self.provider = provider or MockLLMProvider()

    def answer(self, prompt):
        # Le pasamos el prompt del usuario y un "system prompt" fijo que
        # define el rol del agente; el proveedor decide cómo responder.
        return self.provider.complete(prompt, system="You are the Fraud Mitigation Agent, a concise workshop assistant.")


### Proveedor opcional: un LLM real

`MockLLMProvider` es el único que el workshop exige. `OpenAICompatibleProvider` es un adaptador opcional para cualquier endpoint compatible con la API de OpenAI (Groq, NVIDIA NIM, Google AI Studio, la propia OpenAI, etc.) — ver el README para opciones gratuitas sin tarjeta de crédito. `FraudAgent` acepta cualquiera de los dos vía `provider=`.

In [ ]:
class OpenAICompatibleProvider:
    """Adaptador opcional: nunca lo exige el camino core. Funciona con la API
    real de OpenAI y con cualquier otro servicio que exponga un endpoint
    compatible con /chat/completions (Groq, NVIDIA NIM, Google AI Studio, un
    servidor local, etc.) — cambiar de proveedor es solo otro
    base_url/api_key/model, sin tocar código. Ver README.md para opciones
    gratuitas sin tarjeta de crédito."""

    def __init__(self, api_key, model, base_url=None):
        if not api_key:
            raise ValueError("LLM_API_KEY is required for the optional provider")
        if not model:
            raise ValueError("LLM_MODEL is required for the optional provider")
        from openai import OpenAI  # import perezoso: solo hace falta si de verdad usás este proveedor
        kwargs = {"api_key": api_key}
        if base_url:
            kwargs["base_url"] = base_url  # sin esto, apunta por defecto a la API de OpenAI
        self.client = OpenAI(**kwargs)
        self.model = model

    def complete(self, prompt, system=None):
        messages = []  # el chat se arma como una lista de mensajes con rol
        if system:
            messages.append({"role": "system", "content": system})  # instrucciones para el modelo
        messages.append({"role": "user", "content": prompt})  # la pregunta o pedido en sí
        # temperature=0: respuestas lo más reproducibles posible (nunca 100%, pero se acerca).
        response = self.client.chat.completions.create(model=self.model, messages=messages, temperature=0)
        return response.choices[0].message.content or ""  # texto de la primera respuesta


### Probemos

In [ ]:
agent = FraudAgent(db)
print(agent.answer("¿Qué es el Fraud Mitigation Agent y cuál es el papel del agente?"))

# Opcional: si configuraste LLM_API_KEY (y LLM_MODEL) como Colab Secret con
# alguno de los proveedores gratuitos del README, probamos un LLM real en
# vez del mock. Si no, seguimos con MockLLMProvider — el workshop no lo exige.
if os.getenv("LLM_API_KEY") and os.getenv("LLM_MODEL"):
    real_provider = OpenAICompatibleProvider(os.getenv("LLM_API_KEY"), os.getenv("LLM_MODEL"), os.getenv("LLM_BASE_URL"))
    real_agent = FraudAgent(db, provider=real_provider)
    print(real_agent.answer("¿Qué es el Fraud Mitigation Agent y cuál es el papel del agente?"))
else:
    print("Sin LLM_API_KEY configurado: seguimos con MockLLMProvider (ver README para opciones gratuitas sin tarjeta de crédito).")
